In [1]:
import pandas as pd
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_cause_and_scenario

In [2]:
location = "india"
vehicle = "rice"

In [3]:
# Parameters
location = "nigeria"
vehicle = "rice"


In [4]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylls.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylls = pd.read_parquet(path)
else:
    pregnancy_ylls = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylls.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_ylls

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,ylls,cause,other_causes,other_causes,10_to_14,invalid,1,baseline,0,141,0.0
1,ylls,cause,other_causes,other_causes,10_to_14,invalid,2,baseline,0,141,0.0
2,ylls,cause,other_causes,other_causes,10_to_14,invalid,3,baseline,0,141,0.0
3,ylls,cause,other_causes,other_causes,10_to_14,invalid,4,baseline,0,141,0.0
4,ylls,cause,other_causes,other_causes,10_to_14,invalid,5,baseline,0,141,0.0
...,...,...,...,...,...,...,...,...,...,...,...
539995,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,1,zero,0,129,0.0
539996,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,2,zero,0,129,0.0
539997,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,3,zero,0,129,0.0
539998,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,4,zero,0,129,0.0


In [6]:
pregnancy_ylls.groupby("scenario").random_seed.nunique()

scenario
baseline        200
intervention    200
zero            200
Name: random_seed, dtype: int64

In [7]:
assert (pregnancy_ylls[pregnancy_ylls.value > 0].entity == "maternal_disorders").all()

In [8]:
pregnancy_ylls_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylls).pipe(
    lambda df: df[df.index.get_level_values("entity") == "maternal_disorders"]
)
pregnancy_ylls_by_scenario

scenario      entity              wealth_quintile
baseline      maternal_disorders  1                  478458.585064
                                  2                  407190.251839
                                  3                  391085.226711
                                  4                  368896.138488
                                  5                  269842.057704
intervention  maternal_disorders  1                  475596.856180
                                  2                  401640.302086
                                  3                  384613.197327
                                  4                  362817.289355
                                  5                  263939.704177
zero          maternal_disorders  1                  478458.585064
                                  2                  407190.251839
                                  3                  391085.226711
                                  4                  368896.138488
            

In [9]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylds = pd.read_parquet(path)
else:
    pregnancy_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

pregnancy_ylds

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,ylds,cause,pregnancy,pregnant,10_to_14,invalid,1,baseline,0,141,0.0
1,ylds,cause,pregnancy,parturition,10_to_14,invalid,1,baseline,0,141,0.0
2,ylds,cause,pregnancy,postpartum,10_to_14,invalid,1,baseline,0,141,0.0
3,ylds,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,1,baseline,0,141,0.0
4,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,10_to_14,invalid,1,baseline,0,141,0.0
...,...,...,...,...,...,...,...,...,...,...,...
1889995,ylds,cause,pregnancy,postpartum,95_plus,severe,5,zero,0,129,0.0
1889996,ylds,cause,maternal_disorders,maternal_disorders,95_plus,severe,5,zero,0,129,0.0
1889997,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,95_plus,severe,5,zero,0,129,0.0
1889998,ylds,cause,all_causes,all_causes,95_plus,severe,5,zero,0,129,0.0


In [10]:
# Pregnancy has no disability, and maternal hemorrhage disability is counted in maternal_disorders
assert (
    pregnancy_ylds[
        pregnancy_ylds.entity.isin(["pregnancy", "maternal_hemorrhage"])
    ].value
    == 0
).all()

In [11]:
pregnancy_ylds_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylds).pipe(
    lambda df: df[
        ~df.index.get_level_values("entity").isin(["pregnancy", "maternal_hemorrhage"])
    ]
)
pregnancy_ylds_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                  46889.726537
                                  2                  50225.606107
                                  3                  37320.508429
                                  4                  40168.547328
                                  5                  19331.581195
              maternal_disorders  1                  25104.845988
                                  2                  20497.632182
                                  3                  19748.462944
                                  4                  19624.124675
                                  5                  15900.865644
intervention  anemia              1                  45177.851852
                                  2                  47622.893241
                                  3                  34506.833493
                                  4                  37196.012969
                          

In [12]:
pregnancy_dalys_by_scenario = pregnancy_ylls_by_scenario.add(
    pregnancy_ylds_by_scenario, fill_value=0
)
pregnancy_dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                   46889.726537
                                  2                   50225.606107
                                  3                   37320.508429
                                  4                   40168.547328
                                  5                   19331.581195
              maternal_disorders  1                  503563.431052
                                  2                  427687.884021
                                  3                  410833.689655
                                  4                  388520.263163
                                  5                  285742.923347
intervention  anemia              1                   45177.851852
                                  2                   47622.893241
                                  3                   34506.833493
                                  4                   37196.012969
            

In [13]:
ylds_path = f"results/rescaled_child_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(ylds_path).is_file():
    assert (pd.read_parquet(ylds_path)['value'] == 0).all()

In [14]:
path = f"results/rescaled_child_results/{vehicle}/{location}/ylls.parquet"

# NOTE: The child_scenario column currently contains only 'baseline'
# because we didn't have any interventions in the child simulation. If
# we add a child intervention that creates another scenario in this
# column, then results from different child scenarios would get added
# together in the call to aggregate_by_cause_and_scenario below, so we'd
# need to change the processing code in that case.
def assert_unique_child_scenario(df):
    assert set(df.child_scenario.unique()) == {'baseline'}
    return df

if pathlib.Path(path).is_file():
    neonatal_ylls = (
        pd.read_parquet(path)
        .pipe(assert_unique_child_scenario)
        .rename(columns={"maternal_scenario": "scenario"})
    )
else:
    # NOTE: This else branch is for processing the Ethiopia results,
    # where no Vivarium sims were run, so the corresponding DALYs should
    # just be 0
    neonatal_ylls = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/ylls.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_ylls

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,input_draw,random_seed,value
0,ylls,cause,other_causes,other_causes,0_to_5_months,Female,1,baseline,baseline,0,81,13185.404688
1,ylls,cause,other_causes,other_causes,0_to_5_months,Female,2,baseline,baseline,0,81,15527.384438
2,ylls,cause,other_causes,other_causes,0_to_5_months,Female,3,baseline,baseline,0,81,14796.092386
3,ylls,cause,other_causes,other_causes,0_to_5_months,Female,4,baseline,baseline,0,81,13330.820326
4,ylls,cause,other_causes,other_causes,0_to_5_months,Female,5,baseline,baseline,0,81,9813.242439
...,...,...,...,...,...,...,...,...,...,...,...,...
23995,ylls,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,intervention,0,71,14449.077502
23996,ylls,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,intervention,0,71,13039.590243
23997,ylls,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,intervention,0,71,14307.775423
23998,ylls,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,intervention,0,71,9194.962857


In [15]:
neonatal_ylls_by_scenario = aggregate_by_cause_and_scenario(neonatal_ylls)
assert (
    neonatal_ylls_by_scenario[
        neonatal_ylls_by_scenario.index.get_level_values("entity") != "other_causes"
    ]
    == 0
).all()
neonatal_ylls_by_scenario = neonatal_ylls_by_scenario[
    neonatal_ylls_by_scenario.index.get_level_values("entity") == "other_causes"
]
neonatal_ylls_by_scenario = (
    neonatal_ylls_by_scenario.reset_index()
    .assign(entity="lbwsg")
    .set_index(neonatal_ylls_by_scenario.index.names)
    .value
)
neonatal_ylls_by_scenario

scenario      entity  wealth_quintile
baseline      lbwsg   1                  1.535284e+07
                      2                  1.589715e+07
                      3                  1.409936e+07
                      4                  1.245904e+07
                      5                  9.475406e+06
intervention  lbwsg   1                  1.534931e+07
                      2                  1.589231e+07
                      3                  1.409159e+07
                      4                  1.245375e+07
                      5                  9.467039e+06
zero          lbwsg   1                  1.535284e+07
                      2                  1.589715e+07
                      3                  1.409936e+07
                      4                  1.245904e+07
                      5                  9.475406e+06
Name: value, dtype: float64

In [16]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_ylds = pd.read_parquet(path)
else:
    non_pregnancy_anemia_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_ylds

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,766.877919,zero
1,Female,0.0,0.019178,2,700.282642,zero
2,Female,0.0,0.019178,3,589.931789,zero
3,Female,0.0,0.019178,4,459.736805,zero
4,Female,0.0,0.019178,5,352.172009,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,269.578844,intervention
746,Male,95.0,125.000000,2,236.472995,intervention
747,Male,95.0,125.000000,3,237.149569,intervention
748,Male,95.0,125.000000,4,229.612771,intervention


In [17]:
# For comparison with previous round of results, we also look at
# WRA and U5
wra_non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[
        (non_pregnancy_anemia_ylds.sex == "Female")
        & (non_pregnancy_anemia_ylds.age_start >= 10)
        & (non_pregnancy_anemia_ylds.age_end <= 55)
    ].assign(entity="anemia", input_draw="draw_0")
)
wra_non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  293489.769842
                      2                  290634.629142
                      3                  270108.007457
                      4                  261096.090280
                      5                  253149.544942
intervention  anemia  1                  284157.553927
                      2                  277619.395999
                      3                  254107.544359
                      4                  242284.696496
                      5                  232911.205322
zero          anemia  1                  293489.769842
                      2                  290634.629142
                      3                  270108.007457
                      4                  261096.090280
                      5                  253149.544942
Name: value, dtype: float64

In [18]:
scenarios[1]

'zero'

In [19]:
(
    wra_non_pregnancy_anemia_ylds_by_scenario.loc["baseline"].sum()
    + pregnancy_ylds_by_scenario.loc[("baseline", "anemia")].sum()
) - (
    wra_non_pregnancy_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
    + pregnancy_ylds_by_scenario.loc[(scenarios[1], "anemia")].sum()
)

0.0

In [20]:
u5_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[(non_pregnancy_anemia_ylds.age_end <= 5)].assign(
        entity="anemia", input_draw="draw_0"
    )
)
u5_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  293493.017208
                      2                  268664.813364
                      3                  197222.364523
                      4                  155609.297594
                      5                  112680.166776
intervention  anemia  1                  284414.500604
                      2                  257492.102113
                      3                  185542.235656
                      4                  144374.890401
                      5                  102983.682296
zero          anemia  1                  293493.017208
                      2                  268664.813364
                      3                  197222.364523
                      4                  155609.297594
                      5                  112680.166776
Name: value, dtype: float64

In [21]:
(
    u5_anemia_ylds_by_scenario.loc["baseline"].sum()
    - u5_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
)

0.0

In [22]:
non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  977586.330847
                      2                  918661.844674
                      3                  755336.324359
                      4                  674418.326204
                      5                  587868.028324
intervention  anemia  1                  946614.522615
                      2                  878360.193811
                      3                  710195.496713
                      4                  625705.124143
                      5                  540134.946567
zero          anemia  1                  977586.330847
                      2                  918661.844674
                      3                  755336.324359
                      4                  674418.326204
                      5                  587868.028324
Name: value, dtype: float64

In [23]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ylls_by_scenario.csv"
if pathlib.Path(path).is_file():
    neural_tube_defect_ylls_by_scenario = pd.read_csv(path)
else:
    neural_tube_defect_ylls_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/intervention/ylls_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

neural_tube_defect_ylls_by_scenario = neural_tube_defect_ylls_by_scenario.set_index(
    ["scenario", "entity", "wealth_quintile"]
).value
neural_tube_defect_ylls_by_scenario

scenario      entity  wealth_quintile
zero          ntd     1                  412740.875039
                      2                  422053.233994
                      3                  382074.313655
                      4                  338670.157838
                      5                  297460.293500
baseline      ntd     1                  412740.875039
                      2                  422053.233994
                      3                  382074.313655
                      4                  338670.157838
                      5                  297460.293500
intervention  ntd     1                  380928.414208
                      2                  363619.591968
                      3                  311748.060989
                      4                  259655.161511
                      5                  221947.934549
Name: value, dtype: float64

In [24]:
dalys_by_scenario = (
    pregnancy_dalys_by_scenario.add(neonatal_ylls_by_scenario, fill_value=0)
    .add(non_pregnancy_anemia_ylds_by_scenario, fill_value=0)
    .add(neural_tube_defect_ylls_by_scenario, fill_value=0)
)
dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                  1.024476e+06
                                  2                  9.688875e+05
                                  3                  7.926568e+05
                                  4                  7.145869e+05
                                  5                  6.071996e+05
              lbwsg               1                  1.535284e+07
                                  2                  1.589715e+07
                                  3                  1.409936e+07
                                  4                  1.245904e+07
                                  5                  9.475406e+06
              maternal_disorders  1                  5.035634e+05
                                  2                  4.276879e+05
                                  3                  4.108337e+05
                                  4                  3.885203e+05
                          

In [25]:
import pathlib

In [26]:
path = f"./results/{location}/{vehicle}/dalys_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
dalys_by_scenario.to_csv(path)